# Preprocesamiento: imputación, escalado y desbalance

Esta notebook no entrena modelos. Deja la matriz lista y el `Pipeline` guardado en `data/preprocesado/pipeline.joblib`.

Tres operaciones que no son lo mismo:

- **Imputación:** qué hacer con lo que falta.
- **Escalado:** poner en la misma escala las columnas numéricas. No cambia cuántos hechos son mortales.
- **Peso de clase:** que los graves y los mortales no desaparezcan al lado de los 62.174 leves. No es un escalado.

## 1. Datos y columnas que entran

Se parte de `data/siniestros_hechos_limpio.csv`. La gravedad, los conteos de víctimas por gravedad y el id quedan afuera: el prefijo `HC`/`LC` ya es la etiqueta. `participantes_siniestro` repite el modo y la contraparte. Latitud y longitud ubican el hecho y no entran a esta matriz. El total de víctimas queda anotado para probarlo más adelante, no en esta primera matriz.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

ROOT = Path.cwd()
for candidato in [ROOT, *ROOT.parents]:
    if (candidato / "data" / "siniestros_hechos_limpio.csv").exists():
        ROOT = candidato
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.escala_hora import EscalaHoraSinImputar

RUTA = ROOT / "data" / "siniestros_hechos_limpio.csv"
SALIDA = ROOT / "data" / "preprocesado"
SALIDA.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 40)

COLUMNAS_NUM = ["mes_siniestro", "dia_siniestro"]
COLUMNA_HORA = "rango_horario"
COLUMNAS_CAT = [
    "comuna_siniestro",
    "tipo_de_via_siniestro",
    "modo_desplazamiento_victima",
    "contraparte_siniestro",
]
FEATURES = COLUMNAS_NUM + [COLUMNA_HORA] + COLUMNAS_CAT
NIVELES = ["LEVE", "GRAVE", "MORTAL"]

df = pd.read_csv(RUTA, sep=";", encoding="utf-8-sig")
X = df[FEATURES].copy()
y = df["gravedad_siniestro"].astype(str)
print(f"Filas: {len(df)}")
print(y.value_counts().reindex(NIVELES).to_string())
print("Nulos en las columnas que entran:")
print(X.isna().sum())


Filas: 65818
gravedad_siniestro
LEVE      62174
GRAVE      2955
MORTAL      689
Nulos en las columnas que entran:
mes_siniestro                   0
dia_siniestro                   0
rango_horario                  81
comuna_siniestro                0
tipo_de_via_siniestro           0
modo_desplazamiento_victima     0
contraparte_siniestro           0
dtype: int64


## 2. Split antes de mirar medias

La media, la mediana, el escalador y las dummies se calculan solo con el entrenamiento. Si se calcularan con todo el archivo, la prueba se filtraría en el preprocesado. El reparto es 80/20 y estratificado, por gravedad_siniestro, para que las dos partes conserven la proporción de los tres niveles.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def conteo(serie):
    return serie.value_counts().reindex(NIVELES).fillna(0).astype(int)

print("Entrenamiento:", len(X_train))
print(conteo(y_train).to_string())
print("Prueba:", len(X_test))
print(conteo(y_test).to_string())
faltante = X["rango_horario"].isna()
print("Horas faltantes en entrenamiento:", int(X_train["rango_horario"].isna().sum()))
print("Horas faltantes en prueba:", int(X_test["rango_horario"].isna().sum()))
print("Mortales sin hora, en todo el archivo:", int((faltante & y.eq("MORTAL")).sum()))


Entrenamiento: 52654
gravedad_siniestro
LEVE      49739
GRAVE      2364
MORTAL      551
Prueba: 13164
gravedad_siniestro
LEVE      12435
GRAVE       591
MORTAL      138
Horas faltantes en entrenamiento: 67
Horas faltantes en prueba: 14
Mortales sin hora, en todo el archivo: 1


## 3. La media de la hora, y por qué no se usa

La media solo está definida para números. En modo, vía, comuna y contraparte el faltante ya es la categoría `SIN_DATO`: no hay una media entre «moto» y «peatón».

En la hora sí se puede calcular. Media y mediana, tomadas del entrenamiento, caen en la tarde. Esa es la franja con más hechos y con la tasa mortal más baja. Rellenar las filas vacías con ese valor les inventa una hora tranquila. De las 81 horas vacías, una es un hecho mortal: asignarle las 14 es tapar el dato, no recuperarlo.

In [3]:
hora_train = X_train["rango_horario"]
media = float(hora_train.mean())
mediana = float(hora_train.median())
print(f"Media de la hora en entrenamiento: {media:.2f}")
print(f"Mediana de la hora en entrenamiento: {mediana:.2f}")

tasa = (
    X_train.assign(es_mortal=y_train.eq("MORTAL").to_numpy())
    .dropna(subset=["rango_horario"])
    .groupby("rango_horario")
    .agg(hechos=("es_mortal", "size"), mortales=("es_mortal", "sum"))
    .assign(tasa=lambda d: d["mortales"] / d["hechos"])
)
print("Tasa mortal en el entrenamiento, horas de la tarde y de la madrugada:")
print(tasa.loc[[3, 4, 5, 13, 14, 15, 16, 17], ["hechos", "mortales", "tasa"]])


Media de la hora en entrenamiento: 13.64
Mediana de la hora en entrenamiento: 14.00
Tasa mortal en el entrenamiento, horas de la tarde y de la madrugada:
               hechos  mortales      tasa
rango_horario                            
3.0               408        26  0.063725
4.0               371        16  0.043127
5.0               598        23  0.038462
13.0             3365        22  0.006538
14.0             3588        34  0.009476
15.0             3432        20  0.005828
16.0             3688        26  0.007050
17.0             3837        26  0.006776


In [4]:
categoricas = []
for columna in COLUMNAS_CAT:
    serie = X_train[columna].astype("string")
    categoricas.append({
        "columna": columna,
        "categorias": int(serie.nunique()),
        "sin_dato": int(serie.eq("SIN_DATO").sum()),
        "media": "no está definida",
    })
pd.DataFrame(categoricas)

,columna,categorias,sin_dato,media
0,comuna_siniestro,16,2864,no está definida
1,tipo_de_via_siniestro,5,10043,no está definida
2,modo_desplazamiento_victima,13,8767,no está definida
3,contraparte_siniestro,16,11273,no está definida


El tratamiento que se guarda es un indicador, `hora_faltante`. Vale 1 cuando la hora no vino en el archivo y 0 cuando sí. No se le escribe la media ni la mediana.

Para que la matriz no tenga huecos, la hora escalada de esas filas queda en 0. Ese 0 es el centro de la escala, no las 14. Una hora real cercana a la media también queda cerca de 0, pero con `hora_faltante` en 0. Las dos filas no son iguales: el indicador las separa.

In [5]:
comparacion = pd.DataFrame([
    {
        "tratamiento": "Media",
        "que_ven_las_filas_sin_hora": f"las {media:.2f}, una hora de la tarde",
        "se_guarda": "no",
    },
    {
        "tratamiento": "Mediana",
        "que_ven_las_filas_sin_hora": f"las {mediana:.0f}, también la tarde",
        "se_guarda": "no",
    },
    {
        "tratamiento": "Indicador hora_faltante",
        "que_ven_las_filas_sin_hora": "ninguna hora de reloj; el indicador vale 1",
        "se_guarda": "sí",
    },
])
comparacion

,tratamiento,que_ven_las_filas_sin_hora,se_guarda
0,Media,"las 13.64, una hora de la tarde",no
1,Mediana,"las 14, también la tarde",no
2,Indicador hora_faltante,ninguna hora de reloj; el indicador vale 1,sí


## 4. Escalado

`StandardScaler` resta la media y divide por el desvío, calculados en el entrenamiento. Se aplica a mes, día y hora. Las dummies de las categorías quedan en 0 y 1: escalarlas no aporta.

El escalador no toca la etiqueta. LEVE, GRAVE y MORTAL no se escalan, y no son una entrada del modelo.

La hora es circular: las 23 y las 0 quedan en puntas opuestas de la recta, aunque en el reloj están a una hora. Multiplicar por una constante no dobla esa recta. Una codificación con seno y coseno resolvería el ciclo; queda anotada y no se arma un segundo pipeline.

In [6]:
categoricas_enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
matriz = ColumnTransformer(
    transformers=[
        ("calendario", StandardScaler(), COLUMNAS_NUM),
        ("hora", EscalaHoraSinImputar(), [COLUMNA_HORA]),
        ("categoricas", categoricas_enc, COLUMNAS_CAT),
    ]
)
pipeline = Pipeline([("matriz", matriz)])
pipeline.fit(X_train, y_train)

train_matriz = pipeline.transform(X_train)
test_matriz = pipeline.transform(X_test)
nombres = list(pipeline.named_steps["matriz"].get_feature_names_out())
print("Columnas de la matriz:", len(nombres))
print("Entrenamiento:", train_matriz.shape, "| prueba:", test_matriz.shape)
print("¿Quedó algún nulo?", bool(np.isnan(train_matriz).any() or np.isnan(test_matriz).any()))

escala_mes = pipeline.named_steps["matriz"].named_transformers_["calendario"]
escala_hora = pipeline.named_steps["matriz"].named_transformers_["hora"]
print(f"Mes, media y desvío del entrenamiento: {escala_mes.mean_[0]:.2f}, {escala_mes.scale_[0]:.2f}")
print(f"Día, media y desvío del entrenamiento: {escala_mes.mean_[1]:.2f}, {escala_mes.scale_[1]:.2f}")
print(f"Hora observada, media y desvío: {escala_hora.mean_:.2f}, {escala_hora.scale_:.2f}")

Columnas de la matriz:

 54
Entrenamiento: (52654, 54) | prueba: (13164, 54)
¿Quedó algún nulo? False
Mes, media y desvío del entrenamiento: 6.74, 3.42
Día, media y desvío del entrenamiento: 15.57, 8.75
Hora observada, media y desvío: 13.64, 5.55


In [7]:
def resumen_numerico(valores, nombre):
    return {
        "columna": nombre,
        "min": float(np.min(valores)),
        "max": float(np.max(valores)),
        "media": float(np.mean(valores)),
        "desvio": float(np.std(valores)),
    }

idx = {nombre: i for i, nombre in enumerate(nombres)}
hora_ok = train_matriz[:, idx["hora__hora_faltante"]] == 0
filas = [
    resumen_numerico(X_train["mes_siniestro"], "mes, antes"),
    resumen_numerico(train_matriz[:, idx["calendario__mes_siniestro"]], "mes, después"),
    resumen_numerico(X_train["dia_siniestro"], "día, antes"),
    resumen_numerico(train_matriz[:, idx["calendario__dia_siniestro"]], "día, después"),
    resumen_numerico(X_train["rango_horario"].dropna(), "hora observada, antes"),
    resumen_numerico(train_matriz[hora_ok, idx["hora__rango_horario_escalado"]], "hora observada, después"),
]
pd.DataFrame(filas).round(3)

,columna,min,max,media,desvio
0,"mes, antes",1.000,12.000,6.745,3.423
1,"mes, después",-1.679,1.535,-0.000,1.000
2,"día, antes",1.000,31.000,15.574,8.749
3,"día, después",-1.666,1.763,-0.000,1.000
4,"hora observada, antes",0.000,23.000,13.640,5.550
5,"hora observada, después",-2.458,1.687,0.000,1.000


## 5. Los tres niveles

El peso que usa scikit-learn para una clase es la cantidad de filas dividida por (número de clases × filas de esa clase). En el entrenamiento el leve queda en 0,353, el grave en 7,424 y el mortal en 31,854. Todas las filas siguen en la matriz: el peso se usa después, al ajustar el modelo, y no borra ni copia siniestros.

La otra idea, igualar los tres niveles al tamaño de la clase mortal, dejaría 2.067 filas. Por eso no se guarda.

In [8]:
pesos = compute_class_weight("balanced", classes=np.array(NIVELES), y=y_train)
n_train = len(y_train)
conteo_train = y_train.value_counts()
for nivel, peso in zip(NIVELES, pesos):
    n_nivel = int(conteo_train[nivel])
    print(f"Peso {nivel}: {n_train} / (3 × {n_nivel}) = {peso:.3f}")

conteo_todo = y.value_counts()
print()
print("Igualar las tres clases al tamaño de la más chica, la mortal,")
print(f"dejaría {int(conteo_todo['MORTAL']) * 3} filas en el archivo entero")
print(f"y descartaría {int(len(y) - conteo_todo['MORTAL'] * 3)}.")


Peso LEVE: 52654 / (3 × 49739) = 0.353
Peso GRAVE: 52654 / (3 × 2364) = 7.424
Peso MORTAL: 52654 / (3 × 551) = 31.854

Igualar las tres clases al tamaño de la más chica, la mortal,
dejaría 2067 filas en el archivo entero
y descartaría 63751.


## 6. Lo que queda guardado

El pipeline está ajustado con el entrenamiento. La prueba solo se transforma. Para volver a cargarlo, el directorio del proyecto tiene que estar en el path de Python, porque la hora usa `src.escala_hora`.

In [9]:
ruta_pipeline = SALIDA / "pipeline.joblib"
joblib.dump(pipeline, ruta_pipeline)
(SALIDA / "columnas_finales.txt").write_text("\n".join(nombres) + "\n", encoding="utf-8")

conteo_train = y_train.value_counts()
conteo_todo = y.value_counts()
resumen = {
    "filas": int(len(df)),
    "niveles": {nivel: int(conteo_todo[nivel]) for nivel in NIVELES},
    "entrenamiento": int(len(X_train)),
    "prueba": int(len(X_test)),
    "entrenamiento_por_nivel": {nivel: int(conteo_train[nivel]) for nivel in NIVELES},
    "media_hora_entrenamiento": media,
    "mediana_hora_entrenamiento": mediana,
    "pesos": {nivel: float(peso) for nivel, peso in zip(NIVELES, pesos)},
    "columnas_matriz": len(nombres),
}
(SALIDA / "pesos_clase.json").write_text(
    json.dumps(
        {
            "formula": "n / (3 * n_clase), calculado en el entrenamiento",
            "niveles": NIVELES,
            "pesos": {nivel: float(peso) for nivel, peso in zip(NIVELES, pesos)},
        },
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
(SALIDA / "resumen.json").write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

cargado = joblib.load(ruta_pipeline)
retransformado = cargado.transform(X_test)
assert retransformado.shape == test_matriz.shape
assert np.allclose(retransformado, test_matriz)
print("Guardado en", ruta_pipeline)
print("Columnas:", len(nombres))
print("El pipeline recargado reproduce la matriz de prueba.")
print(json.dumps(resumen, ensure_ascii=False, indent=2))


Guardado en D:\Cositas\Progra\sistemas-inteligentes-uni\siniestros-viales-predictor\data\preprocesado\pipeline.joblib
Columnas: 54
El pipeline recargado reproduce la matriz de prueba.
{
  "filas": 65818,
  "niveles": {
    "LEVE": 62174,
    "GRAVE": 2955,
    "MORTAL": 689
  },
  "entrenamiento": 52654,
  "prueba": 13164,
  "entrenamiento_por_nivel": {
    "LEVE": 49739,
    "GRAVE": 2364,
    "MORTAL": 551
  },
  "media_hora_entrenamiento": 13.639834940194344,
  "mediana_hora_entrenamiento": 14.0,
  "pesos": {
    "LEVE": 0.35286864097254333,
    "GRAVE": 7.424421883812747,
    "MORTAL": 31.85359951603146
  },
  "columnas_matriz": 54
}
